# 04 Machine Learning Pipeline & Revenue Risk Framework

## Portfolio Project: Customer Retention & Revenue Risk Analytics

### Overview
This notebook builds stratified train/test ML pipelines using `ColumnTransformer`, trains Logistic Regression and Random Forest models, compares metrics (Accuracy, Precision, Recall, F1, ROC-AUC), extracts feature coefficients & importances, quantifies expected monthly revenue at risk, and constructs the 4-quadrant Customer Risk Matrix.

---


In [ ]:
import sys, os
import pandas as pd
import numpy as np

sys.path.append(r'c:\Users\Atul\Desktop\project-2\customer-retention-revenue-risk-analytics')

from src.data_preprocessing import load_raw_data, clean_customer_data
from src.feature_engineering import engineer_features
from src.churn_model import prepare_data, train_logistic_regression, train_random_forest, predict_churn_probabilities
from src.evaluation import (
    evaluate_model_performance, build_customer_risk_matrix,
    extract_feature_coefficients, extract_random_forest_importance
)

proc_path = r'c:\Users\Atul\Desktop\project-2\customer-retention-revenue-risk-analytics\data\processed\cleaned_telco_churn.csv'
df_proc = pd.read_csv(proc_path)
df_eng = engineer_features(df_proc)

print(f"Engineered Dataset Shape: {df_eng.shape}")


### Train Machine Learning Models
Splitting 80/20 stratified, training Logistic Regression & Random Forest models.


In [ ]:
X_train, X_test, y_train, y_test, preprocessor, cat_cols, num_cols = prepare_data(df_eng)

# Train Logistic Regression
lr_model = train_logistic_regression(X_train, y_train, preprocessor)
lr_probs = predict_churn_probabilities(lr_model, X_test)
lr_preds = (lr_probs >= 0.5).astype(int)
lr_metrics = evaluate_model_performance(y_test, lr_preds, lr_probs)

# Train Random Forest
rf_model = train_random_forest(X_train, y_train, preprocessor)
rf_probs = predict_churn_probabilities(rf_model, X_test)
rf_preds = (rf_probs >= 0.5).astype(int)
rf_metrics = evaluate_model_performance(y_test, rf_preds, rf_probs)

model_comparison = pd.DataFrame([lr_metrics, rf_metrics], index=['Logistic Regression', 'Random Forest'])
print("--- Model Performance Comparison ---")
display(model_comparison)


### Feature Importance & Coefficients Extraction
Examining top drivers of churn.


In [ ]:
df_rf_imp = extract_random_forest_importance(rf_model, cat_cols, num_cols)
print("Top 10 Feature Importances (Random Forest):")
display(df_rf_imp.head(10))

df_lr_coef = extract_feature_coefficients(lr_model, cat_cols, num_cols)
print("\nTop 5 Positive Churn Drivers (Logistic Regression):")
display(df_lr_coef.head(5))


### Customer Risk Prioritization Matrix & Revenue Risk Quantification
Generating expected monthly revenue exposure:
$$\text{ExpectedRevenueAtRisk} = \text{MonthlyCharges} \times \text{ChurnProbability}$$


In [ ]:
all_probs = predict_churn_probabilities(rf_model, df_eng[[c for c in cat_cols + num_cols if c in df_eng.columns]])
df_risk = build_customer_risk_matrix(df_eng, all_probs)

print("--- 4-Quadrant Customer Risk Segment Breakdown ---")
risk_summary = df_risk.groupby('RiskSegment').agg(
    CustomerCount=('CustomerID', 'count'),
    AvgMonthlyCharges=('MonthlyCharges', 'mean'),
    TotalMonthlyMRR=('MonthlyCharges', 'sum'),
    TotalRevenueAtRisk=('ExpectedRevenueAtRisk', 'sum'),
    AvgChurnProbability=('ChurnProbability', 'mean')
).reset_index()

display(risk_summary)

print(f"\nTotal Overall Expected Monthly Revenue at Risk: ${df_risk['ExpectedRevenueAtRisk'].sum():,.2f}")
print("\nTop 10 Highest Priority Customers for Immediate Retention Intervention:")
display(df_risk[['CustomerID', 'Contract', 'Tenure', 'MonthlyCharges', 'ChurnProbability', 'ExpectedRevenueAtRisk', 'RiskSegment', 'RetentionPriority', 'PriorityScore']].head(10))
